# NeuroSeg V2 — Patient-level splitting and MRI preprocessing

Reconstructed from the original Kaggle execution transcript. Source: [Cheng brain tumor dataset](https://doi.org/10.6084/m9.figshare.1512427). Run on Kaggle with Internet enabled; no GPU required. Previously observed results: 3,064 image-mask pairs, 233 patients, 2,124/633/307 train/validation/test images, and 163/46/24 patients respectively. All patient overlap checks were zero and all conversions succeeded. These are historical execution results, not a fresh execution of this notebook. This is a research prototype, not a clinical diagnostic system. Keep generated data private and follow source dataset terms.

## 1. Download original MATLAB archives

In [ ]:
import requests
import zipfile
from pathlib import Path

BASE = Path('/kaggle/working/figshare')
BASE.mkdir(parents=True, exist_ok=True)
mat_files = sorted(BASE.rglob('*.mat'))

if len(mat_files) != 3064:
    response = requests.get('https://api.figshare.com/v2/articles/1512427', timeout=60)
    response.raise_for_status()
    metadata = response.json()
    print('Dataset:', metadata['title'])
    for file in metadata['files']:
        if not file['name'].lower().endswith('.zip'):
            continue
        destination = BASE / file['name']
        url = file.get('download_url') or f"https://api.figshare.com/v2/file/download/{file['id']}"
        print('Downloading:', file['name'])
        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with destination.open('wb') as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        with zipfile.ZipFile(destination) as z:
            z.extractall(BASE)
        destination.unlink()
    mat_files = sorted(BASE.rglob('*.mat'))

assert len(mat_files) == 3064
assert len({p.name for p in mat_files}) == 3064
print('Total MATLAB files:', len(mat_files))

## 2. Inspect source data and extract metadata
Original patient identifiers are used only internally and are not printed.

In [ ]:
import h5py
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

with h5py.File(mat_files[0], 'r') as f:
    for key in f['cjdata'].keys():
        print(key, f['cjdata'][key].shape)

In [ ]:
records = []

for path in tqdm(mat_files):
    with h5py.File(path, 'r') as f:
        data = f['cjdata']
        pid = ''.join(chr(int(x)) for x in np.array(data['PID']).flatten() if int(x) != 0).strip()
        label = int(np.array(data['label']).item())
        records.append({
            'filename': path.name, 'patient_id': pid, 'label': label,
            'height': data['image'].shape[0], 'width': data['image'].shape[1]
        })

df = pd.DataFrame(records)
assert len(df) == 3064
assert df['patient_id'].ne('').all()
assert df['patient_id'].nunique() == 233
df['tumor_type'] = df['label'].map({1: 'Meningioma', 2: 'Glioma', 3: 'Pituitary tumor'})
assert df['tumor_type'].notna().all()
print('Images:', len(df), 'Patients:', df['patient_id'].nunique())
print(df['tumor_type'].value_counts())
print(df.groupby('tumor_type')['patient_id'].nunique())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['tumor_type'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('MRI Images per Tumor Type')
axes[0].set_ylabel('Number of Images')
df.groupby('tumor_type')['patient_id'].nunique().plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title('Patients per Tumor Type')
axes[1].set_ylabel('Number of Patients')
for ax in axes:
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 3. Stratified patient-level split
Preserve the sorted file ordering and random seed of the original experiment. Reserve the test set for final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

assert df.groupby('patient_id')['label'].nunique().max() == 1
patients = df[['patient_id', 'label']].drop_duplicates()
train_patients, temp_patients = train_test_split(
    patients, test_size=0.30, random_state=42, stratify=patients['label']
)
val_patients, test_patients = train_test_split(
    temp_patients, test_size=1/3, random_state=42, stratify=temp_patients['label']
)
train_ids = set(train_patients['patient_id'])
val_ids = set(val_patients['patient_id'])
test_ids = set(test_patients['patient_id'])
split_map = {pid: split for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)] for pid in ids}
df['split'] = df['patient_id'].map(split_map)
assert df['split'].notna().all()
display(df.groupby('split').agg(total_images=('filename', 'count'), total_patients=('patient_id', 'nunique')))
display(pd.crosstab(df['split'], df['tumor_type']))

In [ ]:
print('Train-Val overlap:', len(train_ids & val_ids))
print('Train-Test overlap:', len(train_ids & test_ids))
print('Val-Test overlap:', len(val_ids & test_ids))
assert not (train_ids & val_ids)
assert not (train_ids & test_ids)
assert not (val_ids & test_ids)
assert len(train_ids | val_ids | test_ids) == df['patient_id'].nunique()
assert df['filename'].is_unique
expected = {'train': (2124, 163), 'val': (633, 46), 'test': (307, 24)}
for split, (n_images, n_patients) in expected.items():
    subset = df[df['split'] == split]
    assert len(subset) == n_images and subset['patient_id'].nunique() == n_patients, f'Split differs from original: {split}'
print('Patient overlap checks passed; original split counts reproduced.')

## 4. MRI intensity normalization and binary masks

In [ ]:
from PIL import Image

def normalize_mri(image):
    image = image.astype(np.float32)
    if not np.isfinite(image).all():
        raise ValueError('MRI contains invalid intensity values')
    foreground = image[image > 0]
    if foreground.size == 0:
        raise ValueError('Empty MRI image')
    lower, upper = np.percentile(foreground, [1, 99])
    if upper <= lower:
        raise ValueError('Invalid MRI intensity range')
    normalized = np.clip(image, lower, upper)
    normalized = (normalized - lower) / (upper - lower)
    normalized[image == 0] = 0
    return np.round(normalized * 255).astype(np.uint8)

def prepare_mask(mask):
    values = np.unique(mask)
    if not np.isin(values, [0, 1, 255]).all():
        raise ValueError(f'Unexpected mask values: {values}')
    return (mask > 0).astype(np.uint8) * 255

In [ ]:
sample = df[df['split'] == 'train'].sample(1, random_state=42).iloc[0]
with h5py.File(BASE / sample['filename'], 'r') as f:
    original = np.array(f['cjdata']['image']).T
    original_mask = np.array(f['cjdata']['tumorMask']).T

processed = normalize_mri(original)
binary_mask = prepare_mask(original_mask)
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, image, title in zip(axes, [original, processed, binary_mask, processed],
                            ['Original MRI', 'Normalized MRI', 'Binary Mask', 'Tumor Overlay']):
    ax.imshow(image, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
axes[3].imshow(np.ma.masked_where(binary_mask == 0, binary_mask), cmap='autumn', alpha=0.4)
plt.tight_layout()
plt.show()
print('Original intensity range:', original.min(), original.max())
print('Normalized intensity range:', processed.min(), processed.max())
print('Mask values:', np.unique(binary_mask))

## 5. Convert the full dataset to grayscale PNGs
Keep native resolution; resize in the training loader.

In [ ]:
PROCESSED_DIR = Path('/kaggle/working/processed_dataset')
for split in ['train', 'val', 'test']:
    for folder in ['images', 'masks']:
        (PROCESSED_DIR / split / folder).mkdir(parents=True, exist_ok=True)

In [ ]:
errors = []
processed_count = 0

for row in tqdm(df.itertuples(index=False), total=len(df), desc='Preprocessing MRI images'):
    try:
        with h5py.File(BASE / row.filename, 'r') as f:
            image = np.array(f['cjdata']['image']).T
            mask = np.array(f['cjdata']['tumorMask']).T
        if image.shape != mask.shape:
            raise ValueError('Image-mask shape mismatch')
        image = normalize_mri(image)
        mask = prepare_mask(mask)
        filename = Path(row.filename).stem + '.png'
        Image.fromarray(image).save(PROCESSED_DIR / row.split / 'images' / filename)
        Image.fromarray(mask).save(PROCESSED_DIR / row.split / 'masks' / filename)
        processed_count += 1
    except Exception as e:
        errors.append({'filename': row.filename, 'error': str(e)})

print('Successfully processed:', processed_count)
print('Failed files:', len(errors))
if errors:
    display(pd.DataFrame(errors).head(20))
assert processed_count == 3064
assert len(errors) == 0

## 6. Verify all processed pairs

In [ ]:
verification = []
for split in ['train', 'val', 'test']:
    images = {p.name for p in (PROCESSED_DIR / split / 'images').glob('*.png')}
    masks = {p.name for p in (PROCESSED_DIR / split / 'masks').glob('*.png')}
    expected_names = {Path(name).stem + '.png' for name in df.loc[df['split'] == split, 'filename']}
    assert images == masks == expected_names
    verification.append({'Split': split, 'Images': len(images), 'Masks': len(masks), 'Matched pairs': len(images & masks)})
verification_df = pd.DataFrame(verification)
display(verification_df)
print('Total images:', verification_df['Images'].sum())
print('Total masks:', verification_df['Masks'].sum())
print('All dataset verification checks passed!')

## 7. Save the split manifest without original patient identifiers

In [ ]:
public_manifest = df.copy()
patient_aliases = {pid: f'patient_{i:03d}' for i, pid in enumerate(sorted(df['patient_id'].unique()), start=1)}
public_manifest['patient_group'] = public_manifest['patient_id'].map(patient_aliases)
public_manifest['filename'] = public_manifest['filename'].str.replace('.mat', '.png', regex=False)
public_manifest = public_manifest.drop(columns=['patient_id'])
assert len(public_manifest) == 3064
assert public_manifest['patient_group'].nunique() == 233
public_manifest.to_csv(PROCESSED_DIR / 'dataset_manifest.csv', index=False)
print('Manifest saved:', PROCESSED_DIR / 'dataset_manifest.csv')

## 8. Package the dataset for training
Do not commit raw scans, original patient identifiers, or the generated ZIP to GitHub.

In [ ]:
import shutil
archive_path = shutil.make_archive('/kaggle/working/neuroseg_processed_v2', 'zip', root_dir=PROCESSED_DIR)
print('Archive location:', archive_path)
print(f'Archive size: {Path(archive_path).stat().st_size / (1024 ** 2):.2f} MB')